Alignment_and_Deployment_Lab_v2.ipynb — SOLUCIÓN

# Lab: Alignment y Deployment de Modelos Generativos
## De Raw Model a Production-Ready con SFT

### Objetivos del Lab:
1. Entender la progresión de un modelo Base a uno SFT (Supervised Fine-Tuned).
2. Experimentar con técnicas de fine-tuning eficiente (LoRA).
3. Comparar un modelo fine-tuned con recursos limitados contra un modelo oficial.

Autor: [Clemente Henriquez](https://clemente-h.github.io/), actualizado por [Valentin Barriere](https://valbarriere.github.io/)

---
## Setup Inicial

In [1]:
# Instalación de dependencias
!pip install -q transformers datasets peft accelerate bitsandbytes trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.6 MB/s eta 0:00:00


In [2]:
import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Verificar hardware
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Utilidades Globales

In [4]:
def clear_memory():
    """Limpiar memoria GPU y RAM"""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Memoria limpiada")

In [5]:
def get_gpu_memory():
    """Obtener uso actual de memoria GPU"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0

In [6]:
def generate_text(model, tokenizer, prompt, max_length=200, temperature=0.7):
    """
    Generar texto usando el modelo
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            attention_mask=inputs.attention_mask
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text[len(prompt):].strip()

# PARTE 1: El Problema - Los Modelos Base

### 1.1 EJERCICIO 1: Define tus Preguntas de Evaluación — SOLUCIÓN

**Instrucciones:**
1. Crea una lista de 10 preguntas de evaluación en la variable `eval_questions`.
2. Las preguntas deben ser variadas para probar diferentes capacidades (conocimiento, redacción, etc.).

In [25]:
# SOLUCION
# EJERCICIO 1 (SOLUCIÓN): 10 preguntas variadas para cubrir distintas capacidades
eval_questions = [
    "¿Qué es machine learning?",
    "Explica la diferencia entre aprendizaje supervisado y no supervisado.",
    "Escribe un correo formal pidiendo una extensión de plazo para un proyecto.",
    "¿Cuánto es 15 multiplicado por 12?",
    "Dame una receta simple para hacer pan casero.",
    "¿Qué es la tokenización en un modelo de lenguaje?",
    "Resume en 3 oraciones qué es el cambio climático.",
    "Escribe un poema corto sobre el océano.",
    "¿Cuál es la capital de Argentina?",
    "Traduce al inglés: 'Hoy aprendí a entrenar un modelo de lenguaje.'",
]

In [8]:
print("Preguntas de evaluación definidas:")
for i, q in enumerate(eval_questions, 1):
    print(f"{i}. {q}")

Preguntas de evaluación definidas:
1. ¿Qué es machine learning?
2. Explica la diferencia entre aprendizaje supervisado y no supervisado.
3. Escribe un correo formal pidiendo una extensión de plazo para un proyecto.
4. ¿Cuánto es 15 multiplicado por 12?
5. Dame una receta simple para hacer pan casero.
6. ¿Qué es la tokenización en un modelo de lenguaje?
7. Resume en 3 oraciones qué es el cambio climático.
8. Escribe un poema corto sobre el océano.
9. ¿Cuál es la capital de Argentina?
10. Traduce al inglés: 'Hoy aprendí a entrenar un modelo de lenguaje.'


### 1.2 Cargar y Evaluar Modelo Base

In [7]:
model_name_base = "Qwen/Qwen2.5-1.5B"

In [8]:
print(f"Cargando modelo base: {model_name_base}")
tokenizer = AutoTokenizer.from_pretrained(model_name_base, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Cargando modelo base: Qwen/Qwen2.5-1.5B


In [11]:
model_base = AutoModelForCausalLM.from_pretrained(
    model_name_base,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print(f"Modelo cargado. Memoria GPU usada: {get_gpu_memory():.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Modelo cargado. Memoria GPU usada: 3.09 GB


In [23]:
def evaluate_model_manual(model, tokenizer, questions, model_name="Model"):
    print(f"\n{'='*60}\nEVALUACIÓN: {model_name}\n{'='*60}\n")
    results = []
    for i, question in enumerate(questions, 1):
        print(f"[{i}/{len(questions)}] Pregunta: {question}")
        response = generate_text(model, tokenizer, question, max_length=150)
        print(f"Respuesta: {response[:200]}{'...' if len(response) > 200 else ''}")
        print("-" * 60)
        results.append({'question': question, 'response': response, 'model': model_name})
    return pd.DataFrame(results)

In [13]:
# Guardar resultados y limpiar memoria
results_base = evaluate_model_manual(model_base, tokenizer, eval_questions, "BASE")
del model_base
clear_memory()


EVALUACIÓN: BASE

[1/10] Pregunta: ¿Qué es machine learning?


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: - Tecnología - PCWorld
¿Qué es machine learning?
La inteligencia artificial es una tecnología que permite al ordenador aprender a realizar ciertas tareas de forma automática.
Las máquinas pueden apren...
------------------------------------------------------------
[2/10] Pregunta: Explica la diferencia entre aprendizaje supervisado y no supervisado.


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: Además, proporciona ejemplos de algoritmos de aprendizaje supervisado y no supervisado para cada uno. Aprender es un proceso que consiste en adquirir conocimiento y habilidades, que puede hacerse en d...
------------------------------------------------------------
[3/10] Pregunta: Escribe un correo formal pidiendo una extensión de plazo para un proyecto.


...

Respuesta: - Página Web
En las últimas décadas, la temperatura mundial ha aumentado considerablemente. Los científicos han relacionado esta creciente temperatura con el aumento del nivel del mar y el aumento del...
------------------------------------------------------------
[8/10] Pregunta: Escribe un poema corto sobre el océano.


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: El océano, un misterio infinito

En la profundidad, en el mar inmenso, 
Vivimos la noche y el día, 
Los colores pasan, las olas van, 
Y siempre, siempre, estaremos allí.

El océano es una fuente de mi...
------------------------------------------------------------
[9/10] Pregunta: ¿Cuál es la capital de Argentina?


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: La capital de Argentina es Buenos Aires.
------------------------------------------------------------
[10/10] Pregunta: Traduce al inglés: 'Hoy aprendí a entrenar un modelo de lenguaje.'
Respuesta: Hoy aprendí a entrenar un modelo de lenguaje.
------------------------------------------------------------
Memoria limpiada


### 1.3 Análisis del Modelo Base — SOLUCIÓN

**Instrucciones:**
Basado en las respuestas guardadas en `results_base`, completa la siguiente tabla para tu análisis.

In [ ]:
#Ejemplo
evaluation_ejemplo = {
    'question': eval_questions,
    'understood': [False, False, False, True, False, False, False, False, False, False],
    'coherence': [2, 2, 1, 3, 1, 2, 2, 1, 2, 2],
    'helpfulness': [1, 1, 1, 2, 1, 1, 1, 1, 1, 1],
    'observations': [
        'Continúa con texto técnico sin responder', 'No da explicación clara, divaga',
        'No estructura en lista, texto confuso', 'Responde correctamente por casualidad',
        'No sigue formato de email', 'Texto técnico incomprensible',
        'No da receta estructurada', 'Definición vaga y confusa',
        'Mezcla conceptos sin claridad', 'No resume, genera texto irrelevante'
    ]
}

In [28]:
# SOLUCION
# Completado (SOLUCIÓN): en la práctica estos valores deben ajustarse mirando
# las respuestas reales en `results_base`. Un modelo base (no instruido) suele:
# - continuar el prompt como si fuera texto libre en vez de responderlo,
# - no seguir el formato pedido (listas, emails, etc.),
# - dar respuestas incoherentes o irrelevantes.
evaluation_base = {
    'question': eval_questions,
    'understood': [False, False, True, False, True, True, False, True, True, False],
    'coherence': [2, 2, 4, 3, 4, 5, 3, 4, 5, 1],
    'helpfulness': [2, 1, 4, 1, 4, 5, 2, 4, 5, 1],
    'observations': [
        'Simula el formato de un sitio web (PCWorld) y habla de IA en lugar de definir Machine Learning directamente.',
        'Continúa redactando el enunciado de la pregunta solicitando ejemplos en lugar de responderla.',
        'Sigue la instrucción e inicia un correo formal solicitando la extensión de plazo.',
        'Explica qué es la multiplicación pero no calcula ni entrega el resultado final (180).',
        'Responde correctamente iniciando una receta estructurada con los ingredientes.',
        'Entrega una definición clara, precisa y conceptualmente correcta sobre la tokenización.',
        'Genera un fragmento de artículo web sin respetar la restricción de resumir en 3 oraciones.',
        'Escribe un poema corto y coherente sobre el océano.',
        'Responde de manera directa, precisa y correcta ("La capital de Argentina es Buenos Aires.").',
        'Simplemente repite la oración en español sin realizar la traducción al inglés.'
    ]
}
df_eval_base = pd.DataFrame(evaluation_base)
print("\nTABLA DE EVALUACIÓN - MODELO BASE (completada)")
print(df_eval_base.head(10))


TABLA DE EVALUACIÓN - MODELO BASE (completada)
                                            question  understood  coherence  \
0                          ¿Qué es machine learning?       False          2   
1  Explica la diferencia entre aprendizaje superv...       False          2   
2  Escribe un correo formal pidiendo una extensió...        True          4   
3                 ¿Cuánto es 15 multiplicado por 12?       False          3   
4      Dame una receta simple para hacer pan casero.        True          4   
5  ¿Qué es la tokenización en un modelo de lenguaje?        True          5   
6  Resume en 3 oraciones qué es el cambio climático.       False          3   
7            Escribe un poema corto sobre el océano.        True          4   
8                  ¿Cuál es la capital de Argentina?        True          5   
9  Traduce al inglés: 'Hoy aprendí a entrenar un ...       False          1   

   helpfulness                                       observations  
0            2

## PARTE 2: Supervised Fine-Tuning (SFT)

### 2.1 Preparar Dataset y Configuración LoRA

In [15]:
print("Cargando dataset Alpaca...")
dataset = load_dataset("tatsu-lab/alpaca", split="train")
train_dataset = dataset.select(range(1000))
print(f"Usando {len(train_dataset)} ejemplos para SFT")

Cargando dataset Alpaca...
Usando 1000 ejemplos para SFT


In [16]:
def format_alpaca_instruction(sample):
    if sample['input'] and sample['input'].strip():
        prompt = f"""### Instruction:\n{sample['instruction']}\n\n### Input:\n{sample['input']}\n\n### Response:\n{sample['output']}"""
    else:
        prompt = f"""### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"""
    return {"text": prompt}

In [17]:
formatted_dataset = train_dataset.map(format_alpaca_instruction)
print("Dataset formateado.")

Dataset formateado.


In [11]:
lora_config = LoraConfig(
    r=2, # Probar otros valores de 1 a 32 (si tienen tiempo mas),
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
print("Configuración LoRA creada.")

Configuración LoRA creada.


### 2.2 EJERCICIO 2: Experimentar con LoRA Rank — SOLUCIÓN

**Instrucciones:**
Modifica el valor de 'r' en la configuración LoRA y observa cómo cambia el número de parámetros entrenables. Descomenta y ejecuta el siguiente bloque.

In [18]:
# SOLUCION
# EJERCICIO 2 (SOLUCIÓN): la función ya calcula params entrenables/totales para cada r
def count_trainable_params(r_value):
    lora_config_test = LoraConfig(
        r=r_value,
        lora_alpha=r_value * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type=TaskType.CAUSAL_LM)

    model_temp = AutoModelForCausalLM.from_pretrained(
        model_name_base, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)

    model_temp = get_peft_model(model_temp, lora_config_test)
    trainable = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model_temp.parameters())

    del model_temp
    clear_memory()
    return trainable, total

In [ ]:
# SOLUCION
# SOLUCIÓN: se agregan además r=1 y r=64 para ver el rango completo
r_values = [1, 4, 8, 16, 32, 64]
for r in r_values:
    trainable, total = count_trainable_params(r)
    print(f"r={r}: {trainable:,} params entrenables ({100*trainable/total:.2f}%)")

print("\nObservación: el # de params entrenables crece linealmente con r,")
print("pero incluso con r=64 seguimos entrenando <1% de los parámetros totales del modelo.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Memoria limpiada
r=1: 272,384 params entrenables (0.02%)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Memoria limpiada
r=4: 1,089,536 params entrenables (0.07%)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Memoria limpiada
r=8: 2,179,072 params entrenables (0.14%)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### 2.3 Entrenar y Evaluar con SFT

In [ ]:
print("Cargando modelo para SFT...")
model_sft = AutoModelForCausalLM.from_pretrained(model_name_base, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model_sft = get_peft_model(model_sft, lora_config)

In [18]:
training_args = TrainingArguments(
    output_dir="./qwen-sft-lora", per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=1, learning_rate=2e-4, fp16=True, logging_steps=25, save_strategy="epoch",
    optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_steps=50, report_to="none", remove_unused_columns=False,
)

In [19]:
trainer = SFTTrainer(
    model=model_sft,
    train_dataset=formatted_dataset,
    args=training_args,
    formatting_func=lambda x: x["text"],
)

Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [20]:
print("Iniciando entrenamiento SFT...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Entrenamiento completado en {(end_time - start_time)/60:.1f} minutos")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Iniciando entrenamiento SFT...


Step,Training Loss
25,1.633563
50,1.407341


Entrenamiento completado en 1.8 minutos


In [21]:
model_sft.save_pretrained("./qwen-sft-adapters")
tokenizer.save_pretrained("./qwen-sft-adapters")
print("Adapters LoRA guardados en ./qwen-sft-adapters")

Adapters LoRA guardados en ./qwen-sft-adapters


In [26]:
# Guardar resultados y limpiar memoria
results_sft = evaluate_model_manual(model_sft, tokenizer, eval_questions, "SFT")
del model_sft, trainer
clear_memory()

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



EVALUACIÓN: SFT

[1/10] Pregunta: ¿Qué es machine learning?


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: Machine learning es una ramificación del aprendizaje automático y artificial que permite a un sistema aprender a realizar tareas sin ser programado manualmente para ello. Es un proceso en el que un si...
------------------------------------------------------------
[2/10] Pregunta: Explica la diferencia entre aprendizaje supervisado y no supervisado.


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Respuesta: 1. El cambio climático es un proceso natural que ha ocurrido en la Tierra durante millones de años, pero que se ha acelero en los últimos siglos debido a la actividad humana.
2. El cambio climático se...
------------------------------------------------------------
[8/10] Pregunta: Escribe un poema corto sobre el océano.


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: El mar sin límites, un mundo infinito,
Un lugar donde solo la paz y la calma se adoran.
Tus aguas azules, un color que inspira,
Un mar que nos recuerda, el poder y la belleza.

El mar es un lugar dond...
------------------------------------------------------------
[9/10] Pregunta: ¿Cuál es la capital de Argentina?


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respuesta: La capital de Argentina es Buenos Aires.
------------------------------------------------------------
[10/10] Pregunta: Traduce al inglés: 'Hoy aprendí a entrenar un modelo de lenguaje.'
Respuesta: Hoy aprendí a entrenar un modelo de lenguaje. (Este es un ejemplo de traducción, no un texto en inglés.)
------------------------------------------------------------
Memoria limpiada


In [29]:
evaluation_sft = {
    'question': eval_questions,
    'understood': [True, True, True, True, True, True, True, True, True, False],
    'coherence': [4, 5, 5, 5, 4, 5, 4, 5, 5, 2],
    'helpfulness': [4, 5, 3, 5, 3, 5, 4, 5, 5, 1],
    'observations': [
        'Responde directamente definiendo el concepto, aunque presenta una leve redundancia al decir "ramificación del aprendizaje automático".',
        'Comprende el prompt a la perfección y adopta una estructura muy clara de asistente para explicar la diferencia.',
        'Redacta un correo formal impecable con los marcadores de posición adecuados, aunque responde en inglés en vez de español.',
        'Responde de forma limpia, directa y da la respuesta matemática exacta (180).',
        'Adopta un tono adecuado y organiza la receta con claridad, pero las proporciones de los ingredientes no son del todo realistas.',
        'Explica el concepto de tokenización de manera fluida, clara y orientada al usuario.',
        'Respeta la instrucción de resumen estructurando la respuesta en puntos/oraciones numeradas.',
        'Genera un poema corto, bien estructurado y coherente con la temática solicitada.',
        'Respuesta directa, precisa y limpia sin texto irrelevante.',
        'Falla la tarea de traducción; repite la frase en español y agrega una aclaración errónea entre paréntesis.'
    ]
}

df_eval_sft = pd.DataFrame(evaluation_sft)
print("\nTABLA DE EVALUACIÓN - MODELO SFT (completada)")
print(df_eval_sft.head(10))


TABLA DE EVALUACIÓN - MODELO SFT (completada)
                                            question  understood  coherence  \
0                          ¿Qué es machine learning?        True          4   
1  Explica la diferencia entre aprendizaje superv...        True          5   
2  Escribe un correo formal pidiendo una extensió...        True          5   
3                 ¿Cuánto es 15 multiplicado por 12?        True          5   
4      Dame una receta simple para hacer pan casero.        True          4   
5  ¿Qué es la tokenización en un modelo de lenguaje?        True          5   
6  Resume en 3 oraciones qué es el cambio climático.        True          4   
7            Escribe un poema corto sobre el océano.        True          5   
8                  ¿Cuál es la capital de Argentina?        True          5   
9  Traduce al inglés: 'Hoy aprendí a entrenar un ...       False          2   

   helpfulness                                       observations  
0            4 

## PARTE 3: Comparación Final

Ahora, comparemos nuestro modelo SFT con el modelo oficial de Qwen.

### 3.1 Cargar y Evaluar Modelo Instruct Oficial

In [30]:
print("Cargando Qwen2.5-1.5B-Instruct (modelo oficial)...")
model_instruct = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("Modelo oficial cargado.")

Cargando Qwen2.5-1.5B-Instruct (modelo oficial)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo oficial cargado.


In [31]:
# Guardar resultados y limpiar memoria
results_instruct = evaluate_model_manual(model_instruct, tokenizer, eval_questions, "INSTRUCT")
del model_instruct
clear_memory()


EVALUACIÓN: INSTRUCT

[1/10] Pregunta: ¿Qué es machine learning?
Respuesta: - Página 2 de 4
El término machine learning se refiere a una rama o subcarrera dentro del campo de la inteligencia artificial. Su objetivo principal es que los sistemas puedan aprender y mejorar en su...
------------------------------------------------------------
[2/10] Pregunta: Explica la diferencia entre aprendizaje supervisado y no supervisado.
Respuesta: Aprendizaje Supervisado vs No Supervisado: ¿Qué es cada uno?

Aprendizaje Supervisado y Aprendizaje No Supervisado son dos tipos fundamentales de aprendizaje en machine learning, ambos utilizados para...
------------------------------------------------------------
[3/10] Pregunta: Escribe un correo formal pidiendo una extensión de plazo para un proyecto.
Respuesta: Señor/a Director/a, 
Queremos agradecerle su atención y apoyo durante todo el proceso de desarrollo del proyecto. Sin embargo, nos damos cuenta que estamos a la mitad del tiempo asignado para 

In [32]:
evaluation_instruct = {
    'question': eval_questions,
    'understood': [True, True, True, True, True, True, True, True, True, True],
    'coherence': [3, 4, 4, 5, 3, 3, 3, 4, 5, 4],
    'helpfulness': [3, 4, 4, 5, 2, 3, 3, 4, 5, 4],
    'observations': [
        'Responde adecuadamente, pero arrastra artefactos de raspado web ("- Página 2 de 4") y usa el término poco preciso "subcarrera".',
        'Estructura la respuesta al estilo de un artículo de blog y aborda bien la diferencia entre ambos aprendizajes.',
        'Redacta una carta/correo formal impecable en español solicitando la prórroga de tiempo.',
        'Realiza el cálculo de forma exacta (180), formateando la operación correctamente.',
        'Sigue la estructura de receta, pero alucina un ingrediente peligroso ("levadura cáustica") y proporciones absurdas (3 kg de harina para 1 taza de agua).',
        'Explica el concepto correctamente, pero incluye artefactos de navegación web ("Home » ¿Qué es...?").',
        'Inicia con un título/viñeta prescindible y el texto se corta antes de confirmar si cumplió el límite estricto de 3 oraciones.',
        'Escribe un poema creativo, fluido y bien ambientado sobre el océano.',
        'Responde con la capital exacta y complementa con información contextual acertada sobre la ciudad.',
        'Traduce la oración al inglés a la perfección, aunque filtra fragmentos de las instrucciones del sistema ("You are an AI assistant...") al final.'
    ]
}

df_eval_instruct = pd.DataFrame(evaluation_instruct)
print("\nTABLA DE EVALUACIÓN - MODELO INSTRUCTED (completada)")
print(df_eval_instruct.head(10))


TABLA DE EVALUACIÓN - MODELO INSTRUCTED (completada)
                                            question  understood  coherence  \
0                          ¿Qué es machine learning?        True          3   
1  Explica la diferencia entre aprendizaje superv...        True          4   
2  Escribe un correo formal pidiendo una extensió...        True          4   
3                 ¿Cuánto es 15 multiplicado por 12?        True          5   
4      Dame una receta simple para hacer pan casero.        True          3   
5  ¿Qué es la tokenización en un modelo de lenguaje?        True          3   
6  Resume en 3 oraciones qué es el cambio climático.        True          3   
7            Escribe un poema corto sobre el océano.        True          4   
8                  ¿Cuál es la capital de Argentina?        True          5   
9  Traduce al inglés: 'Hoy aprendí a entrenar un ...        True          4   

   helpfulness                                       observations  
0       

### 3.2 EJERCICIO 3: Comparación Triple (Base vs SFT vs Official) — SOLUCIÓN

**Instrucciones:**
1. Ejecuta la celda para ver las respuestas de los tres modelos lado a lado.
2. Compara la calidad, detalle y estilo de tu modelo SFT con el modelo oficial.
3. Reflexiona sobre qué tan cerca llegaste del modelo oficial con recursos limitados.

In [33]:
print("\n" + "="*100)
print("COMPARACIÓN FINAL: BASE vs NUESTRO SFT vs OFFICIAL INSTRUCT")
print("="*100)
print("\nRecuerda: Entrenamos con solo 1000 ejemplos SFT.")
print("El modelo oficial fue entrenado con millones de ejemplos.\n")


COMPARACIÓN FINAL: BASE vs NUESTRO SFT vs OFFICIAL INSTRUCT

Recuerda: Entrenamos con solo 1000 ejemplos SFT.
El modelo oficial fue entrenado con millones de ejemplos.



In [40]:
for i, question in enumerate(eval_questions):
    print(f"\n{'='*100}")
    print(f"Pregunta: {question}")
    print(f"{'='*100}")

    print("\nMODELO BASE:")
    print(results_base.iloc[i]['response'])

    print("\nNUESTRO SFT:")
    print(results_sft.iloc[i]['response'])

    print("\nOFFICIAL INSTRUCT:")
    print(results_instruct.iloc[i]['response'])

    print("-" * 100)


Pregunta: ¿Qué es machine learning?

MODELO BASE:
- Tecnología - PCWorld 
¿Qué es machine learning?
La inteligencia artificial es una tecnología que permite al ordenador aprender a realizar ciertas tareas de forma automática.
Las máquinas pueden apren...

NUESTRO SFT:
Machine learning es una ramificación del aprendizaje automático y artificial que permite a un sistema aprender a realizar tareas sin ser programado manualmente para ello. Es un proceso en el que un sistema realiza una tarea automáticamente a partir de datos no programados. El sistema se aprende a tomar decisiones a partir de los datos que recibe y utiliza técnicas de machine learning para clasificar o predecir datos.

Es un proceso en el que un sistema realiza una tarea automáticamente a partir de datos no programados

¿Cómo funciona un autoentrenamiento? Un autoentrenamiento es un tipo de aprendizaje automático en el que un sistema aprende a tomar decisiones a partir de los datos que recibe de manera automática, sin nec

In [42]:
def generate_evaluation_summary(eval_base, eval_sft, eval_instruct):
    """
    Compara los resultados de evaluación de los tres modelos (Base, SFT e Instruct).

    Acepta diccionarios o pandas DataFrames.
    Devuelve:
      - df_summary: Tabla con promedios y porcentajes generales por modelo.
      - df_detailed: Tabla comparativa detallada pregunta por pregunta.
    """
    # Asegurar que los inputs sean DataFrames
    dfs = {
        'Base': eval_base,
        'SFT': eval_sft,
        'Instruct': eval_instruct
    }

    # 1. Crear la Tabla Resumen General (Métricas agregadas)
    summary_list = []
    for name, df in dfs.items():
        summary_list.append({
            'Modelo': name,
            '% Entendido': f"{(df['understood'].mean() * 100):.1f}%",
            'Coherencia Promedio': round(df['coherence'].mean(), 2),
            'Utilidad Promedio': round(df['helpfulness'].mean(), 2)
        })

    df_summary = pd.DataFrame(summary_list)

    # 2. Crear la Tabla Comparativa Detallada (Pregunta por pregunta)
    df_detailed = pd.DataFrame({
        '#': range(1, len(dfs['Base']) + 1),
        'Pregunta': dfs['Base']['question'],
        'Entendido (Base/SFT/Inst)': (
            dfs['Base']['understood'].map({True: '✓', False: '✗'}) + ' / ' +
            dfs['SFT']['understood'].map({True: '✓', False: '✗'}) + ' / ' +
            dfs['Instruct']['understood'].map({True: '✓', False: '✗'})
        ),
        'Coherencia (Base/SFT/Inst)': (
            dfs['Base']['coherence'].astype(str) + ' -> ' +
            dfs['SFT']['coherence'].astype(str) + ' -> ' +
            dfs['Instruct']['coherence'].astype(str)
        ),
        'Utilidad (Base/SFT/Inst)': (
            dfs['Base']['helpfulness'].astype(str) + ' -> ' +
            dfs['SFT']['helpfulness'].astype(str) + ' -> ' +
            dfs['Instruct']['helpfulness'].astype(str)
        )
    })

    return df_summary, df_detailed

In [ ]:
df_eval_instruct = pd.DataFrame(evaluation_instruct)

In [43]:
# Generar los resúmenes
df_summary, df_detailed = generate_evaluation_summary(
    df_eval_base,
    df_eval_sft,
    df_eval_instruct
)

print("=" * 60)
print("RESUMEN GENERAL DE COMPORTAMIENTO")
print("=" * 60)
print(df_summary.to_string(index=False))

print("\n" + "=" * 60)
print("EVOLUCIÓN DETALLADA POR PREGUNTA (Base -> SFT -> Instruct)")
print("=" * 60)
print(df_detailed.to_string(index=False))

RESUMEN GENERAL DE COMPORTAMIENTO
  Modelo % Entendido  Coherencia Promedio  Utilidad Promedio
    Base       50.0%                  3.3                2.9
     SFT       90.0%                  4.4                4.0
Instruct      100.0%                  3.8                3.7

EVOLUCIÓN DETALLADA POR PREGUNTA (Base -> SFT -> Instruct)
 #                                                                   Pregunta Entendido (Base/SFT/Inst) Coherencia (Base/SFT/Inst) Utilidad (Base/SFT/Inst)
 1                                                  ¿Qué es machine learning?                 ✗ / ✓ / ✓                2 -> 4 -> 3              2 -> 4 -> 3
 2      Explica la diferencia entre aprendizaje supervisado y no supervisado.                 ✗ / ✓ / ✓                2 -> 5 -> 4              1 -> 5 -> 4
 3 Escribe un correo formal pidiendo una extensión de plazo para un proyecto.                 ✓ / ✓ / ✓                4 -> 5 -> 4              4 -> 3 -> 4
 4                                    

### 3.3 Reflexión Final — SOLUCIÓN (ejemplo de respuesta)

**Preguntas para discutir:**
1. ¿Qué tan cerca llegamos del modelo oficial con recursos limitados?
   - *Con solo 1000 ejemplos y LoRA (r=4), el modelo SFT ya deja de "continuar el texto" y empieza a responder
     directamente a la pregunta, algo que el modelo base no hacía. Sigue lejos del Instruct oficial en
     coherencia y detalle, pero el salto de calidad Base→SFT es mucho mayor que el salto SFT→Instruct.*
2. ¿En qué aspectos nuestro modelo SFT es competitivo?
   - *En seguir el formato pedido (listas, pasos, estructura de respuesta) y en tareas simples (aritmética,
     hechos memorizados), el SFT se acerca bastante al Instruct.*
3. ¿Dónde se nota más la diferencia en la escala de entrenamiento?
   - *En tareas que requieren razonamiento largo, seguir instrucciones compuestas/múltiples, o generar texto
     creativo con buen estilo — ahí el Instruct oficial (entrenado con mucho mas ejemplos e alineamiento) es claramente
     superior.*
4. ¿Qué mejorarías si tuvieras más recursos (datos, compute, tiempo)?
   - *Más datos de instruction-tuning (y más variados), más épocas, un r de LoRA más alto, y agregar una etapa
     de alineamiento por preferencias (DPO/RLHF) después del SFT.*

---
## Resumen del Lab

### Lo que aprendimos:
1. **SFT enseña el formato de "instruction-following" de forma eficiente con LoRA.**
2. **La escala de datos importa, pero la técnica es accesible.** Con pocos datos se logran grandes mejoras.

### Próximos Pasos:
- Experimentar con más datos de SFT.
- Probar con un dataset más específico a un dominio.

---
## Limpieza Final

In [ ]:
clear_memory()
print("Lab completado!")
print("Archivos generados:")
print("  - ./qwen-sft-adapters/: Adapters LoRA después de SFT")
print("¡Excelente trabajo!")